# 🧹 Cleaning — and the decisions it forces on you

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** find everything wrong with this file, and — harder — decide what to do about each thing.

Cleaning data has a reputation as the boring bit. It isn't. Almost every cleaning step is a **decision about the biology**, disguised as a decision about code:

- 41 rows have no plant count. Drop those rows, or keep them?
- 33 rows are exact duplicates. Are they data-entry errors, or two real watches that both saw nothing?

pandas will do either. It will not tell you which is right. That part is your job, and it is the part you have to be able to defend in a methods section.

## Run this first

In [ ]:
# Run this first. The raw file, straight off the clipboard.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

print(df.shape)

## Skill 1 — Where are the holes?

`.isna()` gives you a `True`/`False` for every cell. Summed, it gives you a count per column — and this one line should be the second thing you ever run on a new file.

```python
df.isna().sum()
```

`NaN` ("not a number") is pandas' marker for missing. It is contagious: any arithmetic touching it gives `NaN` back. That is deliberate — it stops a missing value silently becoming a zero.

In [ ]:
# DEMO - only one column has holes in it
missing = df.isna().sum()
print(missing[missing > 0])

print()
print("rows affected:", df["Plants"].isna().sum(), "of", len(df))

In [ ]:
# DEMO - who is missing, and does it look random?
gaps = df[df["Plants"].isna()]

print(gaps["Crop_Generic"].value_counts())
print()
print(gaps["Year"].value_counts())

Look at that first table before you touch anything. Two thirds of the missing plant counts are **blackberry** — and blackberry on an allotment is a bramble along a fence, not a countable number of plants. The gap is not random noise. It is a category of crop where the question did not really apply.

That changes what you should do about it, and it is the kind of thing you can only see by looking.

### 🧹 DIY challenge

1. `df["Plants"].mean()` works despite the missing values — pandas skips them. Confirm this by comparing `df["Plants"].count()` with `len(df)`.
2. Build `df["Flowers_Per_Plant"] = df["Total_Flowers"] / df["Plants"]`. How many `NaN`s does the new column have, and why exactly that many?
3. Compare `df.dropna()` with `df.dropna(subset=["Plants"])`. Do they give the same number of rows here? Would they on a messier file?

> 💡 `.count()` counts non-missing values. `len()` counts rows. The difference is your missing data.

In [ ]:
# TODO - your turn

## Skill 2 — Drop, fill, or leave alone

Three options, three different claims about the world:

| what you write | what you are claiming |
|---|---|
| `df.dropna(subset=["Plants"])` | "These watches are unusable for my question." |
| `df["Plants"].fillna(0)` | "There were no plants." — **almost always wrong**, and here it is a lie: something was flowering, or there would be no watch. |
| `df["Plants"].fillna(df["Plants"].median())` | "A typical plot, near enough." — defensible, but you have invented 41 numbers. |
| leave it | "I am not using this column, so it cannot hurt me." |

For this survey the honest answer is the last one. We are counting **insects per flower**, and `Total_Flowers` has no gaps at all. `Plants` is simply not on the path to our result — so we keep the rows and stay away from the column.

Deleting 41 real watches to tidy up a column you were not going to use is a bad trade.

In [ ]:
# DEMO - what each choice costs you
print("rows if we drop:", len(df.dropna(subset=["Plants"])), "of", len(df))

filled = df["Plants"].fillna(df["Plants"].median())
print()
print("median fill changes the mean from",
      round(df["Plants"].mean(), 2), "to", round(filled.mean(), 2))
print("...and the standard deviation from",
      round(df["Plants"].std(), 2), "to", round(filled.std(), 2))

Both numbers moved, and the standard deviation moved **down** — 19.34 to 19.12. Only slightly, because 41 invented rows cannot do much damage among 1,620. But the direction is the point, and it is the direction you get every time: you have added 41 perfectly average plots that nobody ever visited, so the data now looks a little more consistent than the survey actually was.

On a file where a third of the values are missing, that same move is not slight at all. Worth knowing before you reach for `.fillna()` out of habit.

### 🧹 DIY challenge

1. Make a copy `clean = df.copy()` and drop the rows with missing `Plants` from the copy. Check that `df` itself is unchanged.
2. Why `.copy()`? Try `clean = df` instead and change something — what happens to `df`?
3. Write one sentence you could put in a methods section describing what you decided about `Plants` and why.

> 💡 `clean = df` does **not** make a new table. It makes a second name for the same one. `.copy()` makes a new one.

In [ ]:
# TODO - your turn

## Skill 3 — The duplicates that aren't errors

`.duplicated()` marks every row that has appeared before, exactly, in every column.

```python
df.duplicated().sum()        # how many repeats
df[df.duplicated(keep=False)]  # show every member of every repeated group
```

Run it on this file and you get 33. The temptation is enormous: `df.drop_duplicates()`, one line, table is clean, move on.

Do not. Look at them first.

In [ ]:
# DEMO - look before you delete
print("rows flagged as repeats:", df.duplicated().sum())

repeats = df[df.duplicated(keep=False)]
print("rows involved in a repeated group:", len(repeats))

cols = ["Year", "Date", "Allotment", "Plot", "Crop_Generic", "Total_Flowers"] + INSECTS
repeats[cols].head(8)

In [ ]:
# DEMO - what do the repeats have in common?
print(repeats["Allotment"].value_counts())
print()
all_zero = repeats[INSECTS].sum(axis=1) == 0
print("repeats that recorded nothing at all:", all_zero.sum(), "of", len(repeats))

There it is. They are spread across six allotments, but within each repeated group it is the same plot, the same day, the same crop — and **53 of the 59 recorded nothing at all**.

A volunteer watched four separate courgette plants on the same afternoon. Nothing landed on any of them. Four rows, identical in every column, because there was nothing to tell them apart.

Those are four real watches. `drop_duplicates()` would throw three of them away, and every "visits per watch" number you calculate afterwards would be too high, because you would have deleted the empty watches and kept the busy ones.

**The rule:** a duplicate is only an error if you can say *how* it got there. If the rows are identical because the world was identical, they are data.

### 🧹 DIY challenge

1. How many rows would `df.drop_duplicates()` remove? Work out the mean `Bumblebees` before and after. Which direction did it move, and does that match the argument above?
2. `df.duplicated(subset=["Date", "Plot", "Crop_Generic"])` is a looser test — same watch, ignoring the counts. How many does that flag? What would those be?
3. Suppose you found two identical rows with **different** insect counts. Would that be a duplicate, an error, or two real watches? What would you check?

> 💡 `keep=False` marks every member of a group. The default `keep="first"` marks only the repeats, which is what you want for counting but not for looking.

In [ ]:
# TODO - your turn

## Skill 4 — Names that make code readable

`Crop_Generic`, `Total_Flowers`, `Solitary_Bees`. Somebody's spreadsheet conventions, now yours forever unless you change them.

```python
df = df.rename(columns={"Crop_Generic": "Crop", "Total_Flowers": "Flowers"})
```

And the values need the same treatment: `Rbean` and `Bbean` mean nothing to a reader, and will mean nothing to you in six months either.

```python
df["Crop"] = df["Crop"].replace({"Rbean": "Runner bean", "Bbean": "Broad bean"})
```

`.replace()` takes a dictionary of old → new and leaves anything not mentioned alone. `.map()` is its stricter sibling: it replaces everything, and turns anything missing from the dictionary into `NaN`. Use `.replace()` when you are patching a few values, `.map()` when you are translating a whole column and want to be told if you missed one.

In [ ]:
# DEMO - rename the columns, then translate the values
df.columns = df.columns.str.strip()
df = df.rename(columns={
    "Crop_Generic": "Crop",
    "Total_Flowers": "Flowers",
    "Solitary_Bees": "SolitaryBees",
})

df["Crop"] = df["Crop"].replace({"Rbean": "Runner bean", "Bbean": "Broad bean"})

print(df["Crop"].value_counts().head(8))
print()
print(df.columns.tolist())

### 🧹 DIY challenge

1. `Allotment` has a value `Moulsecoomb_Est` and another called `Moulsecoomb`. Look at how many watches each has. Are they one site or two? (You cannot tell from the data alone — say what you would need to ask.)
2. Underscores in site names are ugly in a figure legend. Replace them with spaces for the whole column using `df["Allotment"].str.replace("_", " ")`.
3. Use `.map()` with a dictionary covering only three crops, and look at what happens to the rest. Now you know why `.replace()` exists.

> 💡 `.str.` gives you Python's string methods, applied down a whole column — `.str.lower()`, `.str.strip()`, `.str.contains()`. It is the same `.str` that fixed the header in notebook 1.

In [ ]:
# TODO - your turn

## Skill 5 — A clean table you can hand over 🏆

Cleaning is only finished when somebody else can pick the file up. That means:

1. every step is in the notebook, in order, so it can be re-run from the raw file
2. the result is written out under a **new name** — never overwrite your raw data
3. you can say in one sentence what you changed and why

```python
df.to_csv("pollinators_clean.csv", index=False)
```

`index=False` matters. Without it pandas writes the row numbers as an unnamed first column, and the next person to read the file gets a mystery column called `Unnamed: 0`. You have probably already met one in the wild.

In [ ]:
# DEMO - write it out, and prove it survives the round trip
df.to_csv("pollinators_clean.csv", index=False)

check = pd.read_csv("pollinators_clean.csv")
print("shape matches:", check.shape == df.shape)
print(check.columns.tolist()[:8])

### 🏆 DIY finale — your own cleaning script

Starting again from the raw file, write a single cell that produces a clean table. It should:

1. read the raw CSV
2. strip whitespace from the column names
3. rename `Crop_Generic` → `Crop` and `Total_Flowers` → `Flowers`
4. translate `Rbean` and `Bbean` to their real names
5. add `All_Insects` (the row total across the eight groups) and `Visits_Per_Flower`
6. **keep** the rows with missing `Plants`, and **keep** the duplicates
7. print the final shape and write it to `pollinators_clean.csv`

Then write two sentences underneath saying what you did about the missing values and the duplicates, and why. That paragraph is worth more marks than the code.

> 💡 Write it as one cell that runs top to bottom from the raw file. If you have to run cells out of order to get the right answer, it is not a script, it is a mess.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Working with missing data, in full: <https://pandas.pydata.org/docs/user_guide/missing_data.html>
- Hadley Wickham, *Tidy Data* (2014) — the paper that named the shape all this is heading towards: <https://vita.had.co.nz/papers/tidy-data.pdf>
- Try `df.describe(include="all")`. It adds the text columns, with `unique`, `top` and `freq`.

## Recap

You can now:

- find missing values with `.isna().sum()` and, more importantly, **look at which rows they are**
- choose between dropping, filling and leaving alone — and say what each choice claims
- use `.duplicated()` to find repeats, and **not** delete them reflexively
- rename columns and translate values with `.rename()`, `.replace()`, `.map()` and `.str`
- write a clean file with `to_csv(..., index=False)` and a sentence justifying it

**Next:** `04_cleaning_dates` — the `Date` column is text, the months sort alphabetically, and both of those are about to matter.